In [ ]:
# Install OpenAI API library (used by NVIDIA NIM)
%pip install -U openai

## Imports and Inititalizations

In [1]:
import pandas as pd
import base64
import os
import html
import itertools
import re
import difflib
from openai import OpenAI
from tqdm import tqdm

# Import the evaluator and cleaning functions from your provided files
from doc_parsing_evaluator import ParsingEvaluator, patterns, extract_and_clean_tables
from common import convert_to_halfwidth

# Paths
BASE_DIR = r"D:\Projects\Evaluation Of MultiModal LLMs for Layout Aware Document Parsing\CC-OCR_Dataset\doc_parsing"
OUT_DIR = r"D:\Projects\Evaluation Of MultiModal LLMs for Layout Aware Document Parsing\Evaluation_Results\Gemma3"
MODEL_NAME = "google/gemma-3-27b-it" 

# API Key Logic
env_path = ".env" 
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        API_KEYS = [line.strip() for line in f if line.strip() and not line.startswith("#")]
else:
    API_KEYS = []

key_cycle = itertools.cycle(API_KEYS) if API_KEYS else None
os.makedirs(OUT_DIR, exist_ok=True)

# Initialize Evaluator
evaluator = ParsingEvaluator(group_name="OCR_ReadingOrder_Eval")

## Inference Engine (Nvidea NIM APIs)

In [5]:
def query_nim(image_bytes, task_type):
    if key_cycle is None: return "No API Key"
    client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=next(key_cycle))
    
    if task_type == "table":
        # Focused on Grid Topology and Reading Order
        prompt = (
            "You are a Table Parsing assistant. Extract the table structure into clean HTML. "
            "Use common tags like <body>,<sup,><table>, <tr>, <td>, and <th> tags. Use rowspan/colspan only if cells are merged. "
            "Maintain the exact reading order of cells (left-to-right, row-by-row). "
            "You can show line breaks using //  or you can & wherever it is used in Latex"
            "Do NOT include any CSS or style attributes. Provide only the raw HTML code."
        )
    else:
        # Focused on Semantic Reading Order
        prompt = (
            "You are an expert Document Intelligence assistant. Extract text strictly in its logical reading order. "
            "Use common tags like tabular,<body>,<sup,><table>, <tr>, <td>, and <th> tags. Use rowspan/colspan only if cells are merged. "
            "Do NOT include document boilerplate (like \\documentclass). Use minimal LaTeX tags: "
            "\\section{...} for headers and \\textbf{...} for bold. Use LaTeX ($$) ONLY for math formulas and TABLES "
            "do not overuse latex tags but you may use it wherever you deem necessary/logical"
            "You can show line breaks using //  or you can & wherever it is used in Latex"
            "Output plain text with basic line breaks to reflect the flow. Do not attempt visual layout recreation."
        )

    image_b64 = base64.b64encode(image_bytes).decode("utf-8")
    
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}}
            ]}],
            max_tokens=2048,
            temperature=0.1
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

In [3]:
def generate_inline_diff(gt_text, pred_text):
    """Generates inline HTML diffs focusing ONLY on content mismatches."""
    gt_text = gt_text if gt_text else ""
    pred_text = pred_text if pred_text else ""
    
    matcher = difflib.SequenceMatcher(None, gt_text, pred_text, autojunk=False)
    gt_diff, pred_diff = [], []
    
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        gt_sub = gt_text[i1:i2]
        pred_sub = pred_text[j1:j2]
        
        if tag == 'equal':
            gt_diff.append(html.escape(gt_sub))
            pred_diff.append(html.escape(pred_sub))
        elif tag == 'replace':
            if gt_sub.strip() == pred_sub.strip() and gt_sub.strip() != "":
                gt_diff.append(html.escape(gt_sub))
                pred_diff.append(html.escape(pred_sub))
            else:
                gt_diff.append(f'<mark style="background-color: #ffcccc; color: #900;">{html.escape(gt_sub)}</mark>')
                pred_diff.append(f'<mark style="background-color: #ccffcc; color: #090;">{html.escape(pred_sub)}</mark>')
        elif tag == 'delete':
            if gt_sub.strip(): 
                gt_diff.append(f'<mark style="background-color: #ffcccc; color: #900;">{html.escape(gt_sub)}</mark>')
            else:
                gt_diff.append(html.escape(gt_sub))
        elif tag == 'insert':
            if pred_sub.strip(): 
                pred_diff.append(f'<mark style="background-color: #ccffcc; color: #090;">{html.escape(pred_sub)}</mark>')
            else:
                pred_diff.append(html.escape(pred_sub))
                
    return "".join(gt_diff), "".join(pred_diff)

def clean_for_visualizer(text, task_type="doc"):
    """Applies the same normalization as the evaluator to ensure diffs are fair."""
    if task_type == "table":
        text = extract_and_clean_tables(text)
    else:
        for pattern in patterns:
            text = re.sub(pattern, '', text)
        text = re.sub(r'```(latex|html|text)?', '', text).replace('```', '')
    
    return convert_to_halfwidth(text).strip()

def generate_bucketed_visual_report(flat_results, filepath, task_label, task_type="doc"):
    bucketed_results = {"76-100%": [], "51-75%": [], "26-50%": [], "0-25%": []}
    
    for item in flat_results:
        s = item['score']
        if s > 0.75: bucketed_results["76-100%"].append(item)
        elif s > 0.50: bucketed_results["51-75%"].append(item)
        elif s > 0.25: bucketed_results["26-50%"].append(item)
        else: bucketed_results["0-25%"].append(item)

    title = f"Bucketed Report - {task_label}"
    
    # CSS updated for vertical stacking to prevent table overlap
    head_html = f"""
    <html><head><title>{title}</title>
    <style>
        body {{ font-family: 'Segoe UI', sans-serif; margin: 20px; background-color: #f0f2f5; }}
        .bucket-header {{ background: #2c3e50; color: white; padding: 15px; border-radius: 5px; margin-top: 40px; }}
        .container {{ display: flex; flex-direction: column; gap: 20px; border: 1px solid #ccc; margin-bottom: 40px; padding: 25px; border-radius: 8px; background: white; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }}
        .col-img {{ width: 100%; border-bottom: 2px solid #eee; padding-bottom: 15px; }}
        .col-img img {{ max-height: 400px; max-width: 100%; object-fit: contain; display: block; margin-top: 10px; border: 1px solid #ddd; }}
        .col-code {{ width: 100%; background: #272822; color: #f8f8f2; padding: 15px; border-radius: 5px; overflow-x: auto; box-sizing: border-box; }}
        .col-render {{ width: 100%; background: #ffffff; color: #333; padding: 15px; border: 1px solid #ccc; border-radius: 5px; overflow-x: auto; box-sizing: border-box; }}
        pre {{ white-space: pre-wrap; font-family: 'Consolas', monospace; font-size: 13px; margin: 0; }}
        .score-tag {{ font-weight: bold; color: white; background: #007bff; padding: 6px 12px; border-radius: 15px; font-size: 15px; display: inline-block; margin-bottom: 10px; }}
        table {{ border-collapse: collapse; width: max-content; font-size: 13px; background: white; }}
        th, td {{ border: 1px solid #444; padding: 6px 10px; text-align: left; }}
        details {{ margin-top: 15px; cursor: pointer; color: #0056b3; font-weight: bold; padding: 10px; background: #f8f9fa; border: 1px solid #ddd; border-radius: 5px; }}
        mark {{ padding: 2px; border-radius: 3px; font-weight: bold; }}
    </style></head><body><h1>{title}</h1>
    """

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(head_html)
        for b_range in ["76-100%", "51-75%", "26-50%", "0-25%"]:
            items = bucketed_results[b_range]
            f.write(f"<h2 class='bucket-header'>Bucket: {b_range} ({len(items)} items)</h2>")
            
            for item in items:
                gt_diff, pred_diff = generate_inline_diff(item['gt_vis'], item['pred_vis'])
                
                if task_type == "table":
                    gt_display = f'<div class="col-render"><strong style="font-size: 16px; color: #d9534f;">GROUND TRUTH HTML</strong><br>{item["gt"]}<details><summary>Click to view Raw HTML Diffs</summary><pre style="color:#333; background:#eee; padding:10px;">{gt_diff}</pre></details></div>'
                    pred_display = f'<div class="col-render"><strong style="font-size: 16px; color: #5cb85c;">PREDICTION HTML</strong><br>{item["pred"]}<details><summary>Click to view Raw HTML Diffs</summary><pre style="color:#333; background:#eee; padding:10px;">{pred_diff}</pre></details></div>'
                else:
                    gt_display = f'<div class="col-code"><strong style="color: #ae81ff; font-size: 16px;">GT (RAW TEXT)</strong><pre>{gt_diff}</pre></div>'
                    pred_display = f'<div class="col-code" style="background: #1e1e1e;"><strong style="color: #66d9ef; font-size: 16px;">PRED (RAW TEXT)</strong><pre>{pred_diff}</pre></div>'

                f.write(f"""
                <div class="container">
                    <div class="col-img">
                        <span class="score-tag">Score: {item['score']:.4f}</span>
                        <h3 style="margin: 5px 0;">Image #{item['index']}</h3>
                        <img src="data:image/png;base64,{item['image_b64']}" />
                    </div>
                    {gt_display}
                    {pred_display}
                </div>
                """)
        f.write("</body></html>")

## Document Parsing Evaluation (LaTeX)

In [6]:
import os
import re
import base64
import pandas as pd
from tqdm import tqdm

doc_tasks = {
    "Scanned_Docs": os.path.join(BASE_DIR, "doc", "doc_scan_eng_75.tsv"),
    "Photoed_Docs": os.path.join(BASE_DIR, "doc", "doc_photo_eng_75.tsv")
}

all_doc_results = [] # Master list to track ALL document results

for label, path in doc_tasks.items():
    if not os.path.exists(path): 
        print(f"Skipping {label}: Path not found.")
        continue
        
    df = pd.read_csv(path, sep='\t')
    current_label_results = [] 
    
    print(f"Visualizing {label}...")
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {label}"):
        img_b64 = row['image']
        gt_code = str(row['answer'])
        
        pred_code = query_nim(base64.b64decode(img_b64), "doc")
        score = evaluator.evaluate_single_doc_sample(gt_code, pred_code)
        
        gt_vis = clean_for_visualizer(gt_code, "doc")
        pred_vis = clean_for_visualizer(pred_code, "doc")
        
        result_dict = {
            'image_b64': img_b64, 
            'gt_vis': gt_vis,         
            'pred_vis': pred_vis,     
            'gt': gt_code,            
            'pred': pred_code,        
            'index': f"{label}-{idx}", 
            'score': score
        }
        
        current_label_results.append(result_dict)
        all_doc_results.append(result_dict) # Append to master list
    
    out_path = os.path.join(OUT_DIR, f"bucketed_visual_{label}.html")
    generate_bucketed_visual_report(current_label_results, out_path, task_label=label, task_type="doc")
    print(f"Report saved: {out_path}")

Visualizing Scanned_Docs...


Processing Scanned_Docs: 100%|██████████| 75/75 [35:37<00:00, 28.50s/it]


Report saved: D:\Projects\Evaluation Of MultiModal LLMs for Layout Aware Document Parsing\Evaluation_Results\Gemma3\bucketed_visual_Scanned_Docs.html
Visualizing Photoed_Docs...


Processing Photoed_Docs: 100%|██████████| 75/75 [24:11<00:00, 19.35s/it]


Report saved: D:\Projects\Evaluation Of MultiModal LLMs for Layout Aware Document Parsing\Evaluation_Results\Gemma3\bucketed_visual_Photoed_Docs.html


## Table Parsing Evaluation (HTML)

In [4]:
table_tasks = {
    "Scanned_Tables": os.path.join(BASE_DIR, "table", "table_scan_eng_75.tsv"),
    "Photoed_Tables": os.path.join(BASE_DIR, "table", "table_photo_eng_75.tsv")
}

all_table_results = [] # Master list to track ALL table results

for label, path in table_tasks.items():
    if not os.path.exists(path): 
        print(f"Skipping {label}: Path not found.")
        continue
    
    df = pd.read_csv(path, sep='\t')
    current_label_results = [] 
    
    print(f"Visualizing {label}...")
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {label}"):
        img_b64 = row['image']
        gt_code = str(row['answer'])
        
        pred_code = query_nim(base64.b64decode(img_b64), "table")
        score = evaluator.evaluate_single_table_sample(gt_code, pred_code)
        
        gt_vis = clean_for_visualizer(gt_code, "table")
        pred_vis = clean_for_visualizer(pred_code, "table")
        
        result_dict = {
            'image_b64': img_b64, 
            'gt_vis': gt_vis,         
            'pred_vis': pred_vis,     
            'gt': gt_code,            
            'pred': pred_code,        
            'index': f"{label}-{idx}", 
            'score': score
        }
        
        current_label_results.append(result_dict)
        all_table_results.append(result_dict) # Append to master list
    
    out_path = os.path.join(OUT_DIR, f"bucketed_visual_{label}.html")
    generate_bucketed_visual_report(current_label_results, out_path, task_label=label, task_type="table")
    print(f"Table report saved: {out_path}")

Visualizing Scanned_Tables...


Processing Scanned_Tables: 100%|██████████| 75/75 [29:37<00:00, 23.69s/it]


Table report saved: D:\Projects\Evaluation Of MultiModal LLMs for Layout Aware Document Parsing\Evaluation_Results\Gemma3\bucketed_visual_Scanned_Tables.html
Visualizing Photoed_Tables...


Processing Photoed_Tables: 100%|██████████| 75/75 [23:43<00:00, 18.98s/it]


Table report saved: D:\Projects\Evaluation Of MultiModal LLMs for Layout Aware Document Parsing\Evaluation_Results\Gemma3\bucketed_visual_Photoed_Tables.html


## Final Report 

In [7]:
import json

# --- FINAL REPORT GENERATION ---
report_path = os.path.join(OUT_DIR, "stage1_new_final_report.json")
all_reports = []

# Helper to calculate metrics from the results list
def calculate_metrics(results):
    if not results:
        return 0, {"76-100%": 0, "51-75%": 0, "26-50%": 0, "0-25%": 0}
    
    avg_score = sum(item['score'] for item in results) / len(results)
    
    # Calculate distribution based on the buckets
    distributions = {"76-100%": 0, "51-75%": 0, "26-50%": 0, "0-25%": 0}
    for item in results:
        s = item['score']
        if s > 0.75: distributions["76-100%"] += 1
        elif s > 0.50: distributions["51-75%"] += 1
        elif s > 0.25: distributions["26-50%"] += 1
        else: distributions["0-25%"] += 1
    
    return avg_score, distributions

# Use the global aggregated lists we created in the loops above
doc_score, doc_dist = calculate_metrics(all_doc_results)
table_score, table_dist = calculate_metrics(all_table_results)

# Load existing results if the file already exists
if os.path.exists(report_path):
    with open(report_path, "r") as f:
        try:
            existing_data = json.load(f)
            if isinstance(existing_data, list):
                all_reports = existing_data
            elif isinstance(existing_data, dict):
                all_reports.append(existing_data)
        except json.JSONDecodeError:
            pass

# Construct the comprehensive summary
summary = {
    "Model": MODEL_NAME,
    "Track": "English Document Parsing",
    "Quantitative_Results": {
        "Document_Parsing_NED_Avg": round(doc_score, 4),
        "Table_Parsing_TEDS_Avg": round(table_score, 4)
    },
    "Qualitative_Buckets": {
        "Documents": doc_dist,
        "Tables": table_dist
    }
}

# Check if model already exists in the report to update it, otherwise append
updated = False
for i, report in enumerate(all_reports):
    if report.get("Model") == MODEL_NAME:
        all_reports[i] = summary
        updated = True
        break

if not updated:
    all_reports.append(summary)

# Save to JSON
with open(report_path, "w") as f:
    json.dump(all_reports, f, indent=4)

print(f"Evaluation Complete! Summary saved to: {report_path}")
print("\n--- Summary of Performance ---")
print(f"Total Documents Evaluated: {len(all_doc_results)}")
print(f"Doc Average Score: {doc_score:.4f}")
print(f"Total Tables Evaluated: {len(all_table_results)}")
print(f"Table Average Score: {table_score:.4f}")

Evaluation Complete! Summary saved to: D:\Projects\Evaluation Of MultiModal LLMs for Layout Aware Document Parsing\Evaluation_Results\Gemma3\stage1_new_final_report.json

--- Summary of Performance ---
Total Documents Evaluated: 150
Doc Average Score: 0.5202
Total Tables Evaluated: 150
Table Average Score: 0.6199


In [8]:
import json
import os

# --- FINAL REPORT GENERATION ---
report_path = os.path.join(OUT_DIR, "stage1_new_final_report.json")
all_reports = []

# Helper to calculate metrics from the results list
def calculate_metrics(results):
    if not results:
        return 0, {"76-100%": 0, "51-75%": 0, "26-50%": 0, "0-25%": 0}
    
    avg_score = sum(item['score'] for item in results) / len(results)
    
    # Calculate distribution based on the buckets
    distributions = {"76-100%": 0, "51-75%": 0, "26-50%": 0, "0-25%": 0}
    for item in results:
        s = item['score']
        if s > 0.75: distributions["76-100%"] += 1
        elif s > 0.50: distributions["51-75%"] += 1
        elif s > 0.25: distributions["26-50%"] += 1
        else: distributions["0-25%"] += 1
    
    return avg_score, distributions

# Separate the aggregated lists based on the label in the index
scanned_docs = [res for res in all_doc_results if res['index'].startswith('Scanned')]
photoed_docs = [res for res in all_doc_results if res['index'].startswith('Photoed')]

scanned_tables = [res for res in all_table_results if res['index'].startswith('Scanned')]
photoed_tables = [res for res in all_table_results if res['index'].startswith('Photoed')]

# Calculate metrics for documents
doc_score_all, doc_dist_all = calculate_metrics(all_doc_results)
doc_score_scan, doc_dist_scan = calculate_metrics(scanned_docs)
doc_score_photo, doc_dist_photo = calculate_metrics(photoed_docs)

# Calculate metrics for tables
table_score_all, table_dist_all = calculate_metrics(all_table_results)
table_score_scan, table_dist_scan = calculate_metrics(scanned_tables)
table_score_photo, table_dist_photo = calculate_metrics(photoed_tables)

# Load existing results if the file already exists
if os.path.exists(report_path):
    with open(report_path, "r") as f:
        try:
            existing_data = json.load(f)
            if isinstance(existing_data, list):
                all_reports = existing_data
            elif isinstance(existing_data, dict):
                all_reports.append(existing_data)
        except json.JSONDecodeError:
            pass

# Construct the comprehensive summary
summary = {
    "Model": MODEL_NAME,
    "Track": "English Document Parsing",
    "Quantitative_Results": {
        "Document_Parsing_Overall_Avg": round(doc_score_all, 4),
        "Document_Parsing_Scanned_Avg": round(doc_score_scan, 4),
        "Document_Parsing_Photoed_Avg": round(doc_score_photo, 4),
        "Table_Parsing_Overall_Avg": round(table_score_all, 4),
        "Table_Parsing_Scanned_Avg": round(table_score_scan, 4),
        "Table_Parsing_Photoed_Avg": round(table_score_photo, 4)
    },
    "Qualitative_Buckets": {
        "Documents_Overall": doc_dist_all,
        "Documents_Scanned": doc_dist_scan,
        "Documents_Photoed": doc_dist_photo,
        "Tables_Overall": table_dist_all,
        "Tables_Scanned": table_dist_scan,
        "Tables_Photoed": table_dist_photo
    }
}

# Check if model already exists in the report to update it, otherwise append
updated = False
for i, report in enumerate(all_reports):
    if report.get("Model") == MODEL_NAME:
        all_reports[i] = summary
        updated = True
        break

if not updated:
    all_reports.append(summary)

# Save to JSON
with open(report_path, "w") as f:
    json.dump(all_reports, f, indent=4)

print(f"Evaluation Complete! Summary saved to: {report_path}")
print("\n--- Summary of Performance ---")
print(f"Total Documents Evaluated: {len(all_doc_results)} (Scanned: {len(scanned_docs)}, Photoed: {len(photoed_docs)})")
print(f"  > Overall Doc Avg: {doc_score_all:.4f}")
print(f"  > Scanned Doc Avg: {doc_score_scan:.4f}")
print(f"  > Photoed Doc Avg: {doc_score_photo:.4f}")
print(f"\nTotal Tables Evaluated: {len(all_table_results)} (Scanned: {len(scanned_tables)}, Photoed: {len(photoed_tables)})")
print(f"  > Overall Table Avg: {table_score_all:.4f}")
print(f"  > Scanned Table Avg: {table_score_scan:.4f}")
print(f"  > Photoed Table Avg: {table_score_photo:.4f}")

Evaluation Complete! Summary saved to: D:\Projects\Evaluation Of MultiModal LLMs for Layout Aware Document Parsing\Evaluation_Results\Gemma3\stage1_new_final_report.json

--- Summary of Performance ---
Total Documents Evaluated: 150 (Scanned: 75, Photoed: 75)
  > Overall Doc Avg: 0.5202
  > Scanned Doc Avg: 0.4684
  > Photoed Doc Avg: 0.5719

Total Tables Evaluated: 150 (Scanned: 75, Photoed: 75)
  > Overall Table Avg: 0.6199
  > Scanned Table Avg: 0.6929
  > Photoed Table Avg: 0.5469
